# VERIFY_01_FFDRI_결과확인 v3

`calc_ffdri.py` 결과를 검증하고, 시군구별 `mean_ffdri`가 왜 높거나 낮은지 `DWI/FMI/TMI`로 분해하는 노트북입니다.

v3 핵심 추가 내용:

- 결과 parquet 내부에 `month` 컬럼이 없어도 동작합니다.
- `date`에서 `month_key = YYYY-MM`을 생성합니다.
- 시군구별 `mean_ffdri`, `mean_dwi`, `mean_fmi`, `mean_tmi`, `mean_elevation`, `mean_slope`를 비교합니다.
- 동해·양양·강릉 / 평창·양구·인제 비교 섹션을 포함합니다.
- 전신주가 있는 격자만 따로 요약합니다.

권장 저장 위치:

```text
weather/notebooks/VERIFY_01_FFDRI_결과확인_v3.ipynb
```

In [ ]:
from pathlib import Path
import calendar
import warnings

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()

MASTER_GANGWON_PATH = PROJECT_ROOT / "processed" / "master_grid_gangwon.parquet"
MASTER_ORIGINAL_PATH = PROJECT_ROOT / "processed" / "master_grid.parquet"
MASTER_PATH = MASTER_GANGWON_PATH if MASTER_GANGWON_PATH.exists() else MASTER_ORIGINAL_PATH
FFDRI_DIR = PROJECT_ROOT / "processed" / "ffdri"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MASTER_PATH :", MASTER_PATH)
print("FFDRI_DIR   :", FFDRI_DIR)
print("master exists:", MASTER_PATH.exists())
print("ffdri exists :", FFDRI_DIR.exists())

if MASTER_PATH == MASTER_ORIGINAL_PATH:
    print("\n[주의] master_grid_gangwon.parquet가 없어서 master_grid.parquet를 사용합니다.")
    print("      강원도 외부 시군구가 섞여 있으면 결과 해석에 영향을 줄 수 있습니다.")

## 1. DuckDB 뷰 생성

`month` 컬럼을 직접 사용하지 않고, `date`에서 `month_key`를 생성합니다.

In [ ]:
parquet_files = sorted(FFDRI_DIR.glob("**/*.parquet"))
print(f"parquet 파일 수: {len(parquet_files):,}")
for p in parquet_files[:10]:
    print("-", p.relative_to(PROJECT_ROOT))
if len(parquet_files) > 10:
    print(f"... 외 {len(parquet_files)-10:,}개")
if not parquet_files:
    raise FileNotFoundError(f"Parquet 파일을 찾지 못했습니다: {FFDRI_DIR}")

con = duckdb.connect()
ffdri_glob = str((FFDRI_DIR / "**" / "*.parquet")).replace("\\", "/")
master_path = str(MASTER_PATH).replace("\\", "/")

con.execute(f'''
    CREATE OR REPLACE TEMP VIEW ffdri_raw AS
    SELECT *
    FROM read_parquet(
        '{ffdri_glob}',
        union_by_name=true,
        hive_partitioning=false
    )
''')

con.execute('''
    CREATE OR REPLACE TEMP VIEW ffdri_base AS
    SELECT
        *,
        strftime(CAST(date AS DATE), '%Y-%m') AS month_key
    FROM ffdri_raw
''')

con.execute(f'''
    CREATE OR REPLACE TEMP VIEW master_base AS
    SELECT *
    FROM read_parquet('{master_path}', union_by_name=true)
''')

sample = con.execute("SELECT * FROM ffdri_base LIMIT 5").df()
display(sample)
print(sample.dtypes)

## 2. 기본 검증

필수 컬럼, 월별 행 수, 결측치, 중복, 공식 재계산을 확인합니다.

In [ ]:
required_cols = [
    "grid_id", "date", "month_key", "dwi", "fmi", "tmi", "day_weight", "ffdri",
    "predwi", "predwi_class", "rne", "rne_temp", "effective_humidity",
    "ta_mean", "hm_mean", "wind_ws_mean", "rn_day_mean",
]
missing_cols = [c for c in required_cols if c not in sample.columns]
print("누락 필수 컬럼:", missing_cols)
if missing_cols:
    warnings.warn(f"필수 컬럼 누락: {missing_cols}")

master_grid_count = con.execute("SELECT COUNT(DISTINCT grid_id) FROM master_base").fetchone()[0]
print(f"master_grid 고유 grid_id 수: {master_grid_count:,}")

row_count_by_month = con.execute('''
    SELECT
        month_key,
        COUNT(*) AS row_count,
        COUNT(DISTINCT grid_id) AS grid_count,
        COUNT(DISTINCT date) AS date_count,
        MIN(CAST(date AS DATE)) AS min_date,
        MAX(CAST(date AS DATE)) AS max_date
    FROM ffdri_base
    GROUP BY month_key
    ORDER BY month_key
''').df()

def days_in_month(month: str) -> int:
    y, m = map(int, month.split("-"))
    return calendar.monthrange(y, m)[1]

row_count_by_month["expected_date_count"] = row_count_by_month["month_key"].map(days_in_month)
row_count_by_month["expected_row_count"] = row_count_by_month["expected_date_count"] * master_grid_count
row_count_by_month["row_count_diff"] = row_count_by_month["row_count"] - row_count_by_month["expected_row_count"]
row_count_by_month["row_count_ok"] = row_count_by_month["row_count_diff"].eq(0)
display(row_count_by_month)

In [ ]:
null_check_cols = [
    "dwi", "fmi", "tmi", "day_weight", "ffdri",
    "predwi", "predwi_class", "rne", "effective_humidity"
]

null_summary = con.execute(f'''
    SELECT
        month_key,
        {", ".join([f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}_nulls" for c in null_check_cols])}
    FROM ffdri_base
    GROUP BY month_key
    ORDER BY month_key
''').df()
display(null_summary)
total_nulls = null_summary.drop(columns=["month_key"]).sum().sum()
print(f"핵심 컬럼 결측치 총합: {total_nulls:,}")

duplicate_summary = con.execute('''
    WITH base AS (
        SELECT month_key, grid_id, date, COUNT(*) AS cnt
        FROM ffdri_base
        GROUP BY month_key, grid_id, date
        HAVING COUNT(*) > 1
    )
    SELECT month_key, COUNT(*) AS duplicated_grid_date_count, SUM(cnt) AS duplicated_rows
    FROM base
    GROUP BY month_key
    ORDER BY month_key
''').df()
display(duplicate_summary)

formula_check = con.execute('''
    SELECT
        MAX(ABS(ffdri - ((7.0 * dwi + 1.5 * fmi + 1.5 * tmi) * day_weight))) AS max_abs_error,
        AVG(ABS(ffdri - ((7.0 * dwi + 1.5 * fmi + 1.5 * tmi) * day_weight))) AS mean_abs_error
    FROM ffdri_base
''').df()
display(formula_check)
max_error = formula_check.loc[0, "max_abs_error"]
if max_error < 1e-8:
    print("FFDRI 공식 검증 통과")
else:
    warnings.warn(f"FFDRI 공식 오차가 큽니다: max_abs_error={max_error}")

## 3. 값 범위 및 월별 요약

In [ ]:
range_cols = [
    "ta_mean", "hm_mean", "wind_ws_mean", "rn_day_mean",
    "effective_humidity", "predwi", "predwi_class",
    "rne_temp", "rne", "dwi", "fmi", "tmi", "day_weight", "ffdri"
]
range_summary = con.execute(f'''
    SELECT
        {", ".join([f"MIN({c}) AS {c}_min, MAX({c}) AS {c}_max, AVG({c}) AS {c}_mean" for c in range_cols])}
    FROM ffdri_base
''').df().T
range_summary.columns = ["value"]
display(range_summary)

monthly_summary = con.execute('''
    SELECT
        month_key,
        COUNT(*) AS n,
        AVG(ffdri) AS mean_ffdri,
        MIN(ffdri) AS min_ffdri,
        QUANTILE_CONT(ffdri, 0.50) AS median_ffdri,
        QUANTILE_CONT(ffdri, 0.90) AS q90_ffdri,
        QUANTILE_CONT(ffdri, 0.95) AS q95_ffdri,
        MAX(ffdri) AS max_ffdri,
        AVG(CASE WHEN ffdri >= 66 THEN 1 ELSE 0 END) AS high_over_66_ratio,
        AVG(CASE WHEN ffdri >= 86 THEN 1 ELSE 0 END) AS extreme_over_86_ratio
    FROM ffdri_base
    GROUP BY month_key
    ORDER BY month_key
''').df()
display(monthly_summary)

## 4. 일별 FFDRI 추세

In [ ]:
daily_summary = con.execute('''
    SELECT
        CAST(date AS DATE) AS date,
        AVG(ffdri) AS mean_ffdri,
        QUANTILE_CONT(ffdri, 0.90) AS q90_ffdri,
        QUANTILE_CONT(ffdri, 0.95) AS q95_ffdri,
        AVG(dwi) AS mean_dwi,
        AVG(day_weight) AS day_weight
    FROM ffdri_base
    GROUP BY date
    ORDER BY date
''').df()
daily_summary["date"] = pd.to_datetime(daily_summary["date"])
display(daily_summary.head())

plt.figure(figsize=(14, 5))
plt.plot(daily_summary["date"], daily_summary["mean_ffdri"], label="mean FFDRI")
plt.plot(daily_summary["date"], daily_summary["q90_ffdri"], label="q90 FFDRI")
plt.plot(daily_summary["date"], daily_summary["q95_ffdri"], label="q95 FFDRI")
plt.title("Daily FFDRI Trend")
plt.xlabel("Date")
plt.ylabel("FFDRI")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 5. 시군구별 컴포넌트 분해

시군구별 `mean_ffdri`가 왜 높거나 낮게 나왔는지 보기 위해 다음 요소를 함께 비교합니다.

- `mean_dwi`: 기상 위험도
- `mean_fmi`: 임상/연료 위험도
- `mean_tmi`: 지형 위험도
- `mean_elevation`: 평균 고도
- `mean_slope`: 평균 경사도
- `forest_ratio`: 산림 격자 비율
- `total_pole_count`: 전신주 개수 합

In [ ]:
city_component_summary = con.execute('''
    WITH f AS (
        SELECT
            grid_id,
            CAST(date AS DATE) AS date,
            month_key,
            ffdri,
            dwi,
            fmi,
            tmi,
            day_weight,
            predwi,
            rne,
            effective_humidity,
            ta_mean,
            hm_mean,
            wind_ws_mean,
            rn_day_mean
        FROM ffdri_base
    ),
    m AS (
        SELECT
            grid_id,
            city_name,
            elevation,
            slope,
            COALESCE(pole_count, 0) AS pole_count,
            is_forest,
            forest_type_code
        FROM master_base
    )
    SELECT
        m.city_name,
        COUNT(*) AS n,
        COUNT(DISTINCT f.grid_id) AS grid_count,
        COUNT(DISTINCT f.date) AS date_count,
        AVG(f.ffdri) AS mean_ffdri,
        QUANTILE_CONT(f.ffdri, 0.50) AS median_ffdri,
        QUANTILE_CONT(f.ffdri, 0.90) AS q90_ffdri,
        MAX(f.ffdri) AS max_ffdri,
        AVG(f.dwi) AS mean_dwi,
        AVG(f.fmi) AS mean_fmi,
        AVG(f.tmi) AS mean_tmi,
        AVG(f.day_weight) AS mean_day_weight,
        AVG(f.predwi) AS mean_predwi,
        AVG(f.rne) AS mean_rne,
        AVG(f.effective_humidity) AS mean_effective_humidity,
        AVG(f.ta_mean) AS mean_ta,
        AVG(f.hm_mean) AS mean_hm,
        AVG(f.wind_ws_mean) AS mean_wind,
        AVG(f.rn_day_mean) AS mean_rain,
        AVG(m.elevation) AS mean_elevation,
        AVG(m.slope) AS mean_slope,
        SUM(m.pole_count) AS total_pole_count,
        AVG(CASE WHEN m.pole_count > 0 THEN 1 ELSE 0 END) AS pole_grid_ratio,
        AVG(CASE WHEN m.is_forest = 1 THEN 1 ELSE 0 END) AS forest_ratio,
        AVG(CASE WHEN f.ffdri >= 66 THEN 1 ELSE 0 END) AS high_over_66_ratio,
        AVG(CASE WHEN f.ffdri >= 86 THEN 1 ELSE 0 END) AS extreme_over_86_ratio
    FROM f
    INNER JOIN m USING (grid_id)
    GROUP BY m.city_name
    ORDER BY mean_ffdri DESC
''').df()
display(city_component_summary)

print("mean_ffdri 상위 5개 시군구")
display(city_component_summary.head(5)[[
    "city_name", "mean_ffdri", "mean_dwi", "mean_fmi", "mean_tmi",
    "mean_elevation", "mean_slope", "mean_effective_humidity",
    "mean_ta", "mean_hm", "mean_wind", "forest_ratio"
]])

print("mean_ffdri 하위 5개 시군구")
display(city_component_summary.tail(5)[[
    "city_name", "mean_ffdri", "mean_dwi", "mean_fmi", "mean_tmi",
    "mean_elevation", "mean_slope", "mean_effective_humidity",
    "mean_ta", "mean_hm", "mean_wind", "forest_ratio"
]])

## 5-1. FFDRI 구성항 기여도 계산

FFDRI 공식:

```text
FFDRI = (7 × DWI + 1.5 × FMI + 1.5 × TMI) × day_weight
```

행 단위 기여도:

```text
DWI contribution = 7 × DWI × day_weight
FMI contribution = 1.5 × FMI × day_weight
TMI contribution = 1.5 × TMI × day_weight
```

In [ ]:
city_contribution_summary = con.execute('''
    WITH f AS (
        SELECT
            grid_id,
            ffdri,
            dwi,
            fmi,
            tmi,
            day_weight,
            (7.0 * dwi * day_weight) AS dwi_contrib,
            (1.5 * fmi * day_weight) AS fmi_contrib,
            (1.5 * tmi * day_weight) AS tmi_contrib
        FROM ffdri_base
    ),
    m AS (
        SELECT grid_id, city_name
        FROM master_base
    )
    SELECT
        m.city_name,
        AVG(f.ffdri) AS mean_ffdri,
        AVG(f.dwi_contrib) AS mean_dwi_contrib,
        AVG(f.fmi_contrib) AS mean_fmi_contrib,
        AVG(f.tmi_contrib) AS mean_tmi_contrib,
        AVG(f.dwi_contrib) / NULLIF(AVG(f.ffdri), 0) AS dwi_contrib_ratio,
        AVG(f.fmi_contrib) / NULLIF(AVG(f.ffdri), 0) AS fmi_contrib_ratio,
        AVG(f.tmi_contrib) / NULLIF(AVG(f.ffdri), 0) AS tmi_contrib_ratio
    FROM f
    INNER JOIN m USING (grid_id)
    GROUP BY m.city_name
    ORDER BY mean_ffdri DESC
''').df()
display(city_contribution_summary)

plot_df = city_contribution_summary.head(10).copy()
x = np.arange(len(plot_df))
bottom = np.zeros(len(plot_df))

plt.figure(figsize=(12, 6))
plt.bar(x, plot_df["mean_dwi_contrib"], label="DWI contribution")
bottom += plot_df["mean_dwi_contrib"].to_numpy()
plt.bar(x, plot_df["mean_fmi_contrib"], bottom=bottom, label="FMI contribution")
bottom += plot_df["mean_fmi_contrib"].to_numpy()
plt.bar(x, plot_df["mean_tmi_contrib"], bottom=bottom, label="TMI contribution")
plt.xticks(x, plot_df["city_name"], rotation=45, ha="right")
plt.ylabel("Mean contribution to FFDRI")
plt.title("Top 10 Cities: FFDRI Component Contributions")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5-2. 동해·양양·강릉 vs 평창·양구·인제 비교

동해·양양·강릉이 높은 이유, 평창·양구·인제가 낮은 이유를 컴포넌트별로 비교합니다.

In [ ]:
high_interest = ["동해", "동해시", "양양", "양양군", "강릉", "강릉시"]
low_interest = ["평창", "평창군", "양구", "양구군", "인제", "인제군"]

interest_summary = city_component_summary[
    city_component_summary["city_name"].isin(high_interest + low_interest)
].copy()

display(interest_summary[[
    "city_name", "mean_ffdri", "mean_dwi", "mean_fmi", "mean_tmi",
    "mean_elevation", "mean_slope", "mean_effective_humidity",
    "mean_ta", "mean_hm", "mean_wind", "forest_ratio", "total_pole_count"
]].sort_values("mean_ffdri", ascending=False))

interest_contrib = city_contribution_summary[
    city_contribution_summary["city_name"].isin(high_interest + low_interest)
].copy()
display(interest_contrib.sort_values("mean_ffdri", ascending=False))

if len(interest_summary) > 0:
    plot_df = interest_summary.sort_values("mean_ffdri", ascending=False).copy()
    x = np.arange(len(plot_df))
    for col in ["mean_ffdri", "mean_dwi", "mean_fmi", "mean_tmi", "mean_elevation", "mean_effective_humidity", "mean_wind"]:
        plt.figure(figsize=(10, 5))
        plt.bar(x, plot_df[col])
        plt.xticks(x, plot_df["city_name"], rotation=45, ha="right")
        plt.ylabel(col)
        plt.title(f"Selected Cities: {col}")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("관심 시군구 이름이 city_name과 매칭되지 않았습니다. city_counts에서 실제 이름을 확인하세요.")

## 6. 전신주 격자 기준 위험도 요약

`pole_count > 0`인 격자만 따로 보면, 전력설비가 실제로 존재하는 격자 중 고위험 지역을 파악할 수 있습니다.

In [ ]:
pole_city_summary = con.execute('''
    WITH f AS (
        SELECT grid_id, CAST(date AS DATE) AS date, month_key, ffdri, dwi, fmi, tmi
        FROM ffdri_base
    ),
    m AS (
        SELECT grid_id, city_name, COALESCE(pole_count, 0) AS pole_count
        FROM master_base
        WHERE COALESCE(pole_count, 0) > 0
    )
    SELECT
        m.city_name,
        COUNT(*) AS n,
        COUNT(DISTINCT f.grid_id) AS pole_grid_count,
        SUM(m.pole_count) AS total_pole_count_repeated_by_date,
        AVG(f.ffdri) AS mean_ffdri,
        QUANTILE_CONT(f.ffdri, 0.90) AS q90_ffdri,
        MAX(f.ffdri) AS max_ffdri,
        AVG(CASE WHEN f.ffdri >= 66 THEN 1 ELSE 0 END) AS high_over_66_ratio,
        AVG(CASE WHEN f.ffdri >= 86 THEN 1 ELSE 0 END) AS extreme_over_86_ratio
    FROM f
    INNER JOIN m USING (grid_id)
    GROUP BY m.city_name
    ORDER BY mean_ffdri DESC
''').df()
display(pole_city_summary)

## 7. 샘플 월 상세 확인

`TARGET_MONTH`를 바꿔가며 특정 월 분포를 확인합니다.

In [ ]:
TARGET_MONTH = "2025-03"

df_month = con.execute('''
    SELECT *
    FROM ffdri_base
    WHERE month_key = $month
''', {"month": TARGET_MONTH}).df()

print(df_month.shape)
display(df_month.head())

display(df_month[["dwi", "fmi", "tmi", "day_weight", "ffdri", "predwi", "rne", "effective_humidity"]].describe().T)

plt.figure(figsize=(10, 5))
plt.hist(df_month["ffdri"].dropna(), bins=50)
plt.title(f"FFDRI Distribution - {TARGET_MONTH}")
plt.xlabel("FFDRI")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Quick Verdict

In [ ]:
issues = []

if not MASTER_PATH.exists():
    issues.append("master_grid parquet 경로가 존재하지 않습니다.")
if not FFDRI_DIR.exists():
    issues.append("processed/ffdri 경로가 존재하지 않습니다.")
if missing_cols:
    issues.append(f"필수 컬럼 누락: {missing_cols}")
if total_nulls > 0:
    issues.append(f"핵심 컬럼 결측치 존재: {total_nulls:,}개")
if not duplicate_summary.empty:
    issues.append("grid_id + date 중복 존재")
if not row_count_by_month["row_count_ok"].all():
    bad_months = row_count_by_month.loc[~row_count_by_month["row_count_ok"], "month_key"].tolist()
    issues.append(f"월별 행 수 불일치: {bad_months}")
if max_error >= 1e-8:
    issues.append(f"FFDRI 공식 오차 존재: max_abs_error={max_error}")

if not issues:
    print("큰 이상은 발견되지 않았습니다.")
else:
    print("점검 필요 항목:")
    for i, issue in enumerate(issues, 1):
        print(f"{i}. {issue}")

## 9. 해석 메모 작성 가이드

나쁜 해석:

```text
평창·양구·인제는 산불 위험이 낮다.
```

권장 해석:

```text
평창·양구·인제는 평균 FFDRI 기준으로 낮게 산출되었다.
이는 고도 높은 산악 격자 비중, TMI 구조, 평균 기상조건 등이 복합적으로 작용한 결과로 해석할 수 있다.
다만 이는 산불 발생·확산 위험이 절대적으로 낮다는 의미는 아니며, 본 지표가 평가하는 '발생 위험 환경' 기준의 결과이다.
```

동해안권 결과는 다음처럼 해석하는 것이 안전합니다.

```text
동해·양양·강릉은 평균 FFDRI 기준 상위권으로 나타났다.
DWI/FMI/TMI 분해 결과를 통해 기상 조건, 침엽수림 비중, 지형 조건 중 어떤 요인이 크게 작용했는지 확인할 수 있다.
```